# Multimodal Video Summarizer (optimised)

Same pipeline as before (Whisper → BERT extractive summary → sentence-to-time mapping → summary video + subtitles), with the slow parts replaced:

| Step | Before | Now |
|---|---|---|
| Audio extraction | MoviePy decodes audio in Python and writes a WAV | FFmpeg streams 16 kHz audio straight into memory |
| Transcription | Hugging Face pipeline on MPS | **mlx-whisper** (Apple's MLX, built for Apple Silicon); HF pipeline kept as fallback on other machines |
| Re-runs | Transcribe every time | Transcript **cached** per video; re-runs skip audio + transcription |
| Summary embeddings | One sentence at a time, on CPU, with gradient tracking | Same embeddings, **batched on the GPU**, no gradients |
| Rendering | MoviePy pipes every frame through Python | **FFmpeg** cuts segments in parallel (hardware encoder on Mac), joins them without re-encoding |
| Subtitles | Second file written after rendering | Added in the same final step |

Every step prints its own time, and the last cell shows a breakdown, so you can see where time goes.

In [10]:
# Run once per environment. Not part of the timed run.
%pip install mlx-whisper transformers torch bert-extractive-summarizer imageio-ffmpeg numpy yt_dlp

Note: you may need to restart the kernel to use updated packages.


## 1. Setup and configuration

In [11]:
import time
from contextlib import contextmanager

STEP_TIMES = {}
RUN_START = time.perf_counter()

@contextmanager
def timed(step):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        STEP_TIMES[step] = time.perf_counter() - t0
        print(f"[{step}] {STEP_TIMES[step]:.1f} s")

with timed("Imports"):
    import bisect, hashlib, json, os, platform, re, shutil, subprocess
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import numpy as np
    import torch
    import imageio_ffmpeg
    from IPython.display import display, Markdown

    # bert-extractive-summarizer still imports TransfoXL classes that transformers 5 removed.
    import transformers
    for _name in ("TransfoXLModel", "TransfoXLTokenizer"):
        try:
            getattr(transformers, _name)
        except (ImportError, AttributeError):
            setattr(transformers, _name, None)
    from summarizer import Summarizer
    from summarizer.transformer_embeddings.bert_embedding import BertEmbedding

# ---------------- Configuration ----------------
VIDEO_PATH = "data/Movie.mp4"
OUTPUT_DIR = Path("output/Movie")

WHISPER_MODEL = "mlx-community/whisper-small-mlx"   # accuracy upgrade: "mlx-community/whisper-large-v3-turbo"
LANGUAGE = "en"                                      # None = auto-detect
SUMMARIZER_MODEL = "bert-large-uncased"              # "bert-base-uncased" is ~3x faster (summaries change slightly)
SUMMARY_RATIO = 0.2
EMBED_BATCH_SIZE = 32

PAD = 0.25            # seconds added before/after each sentence
MERGE_GAP = 1.0       # merge segments closer than this (seconds)
RENDER_WORKERS = 3    # segments cut in parallel
USE_TRANSCRIPT_CACHE = True
# ------------------------------------------------


SUMMARY_RATIO = 0.35          # share of candidate sentences to keep
SUMMARY_NUM_SENTENCES = None  # e.g. 40 to fix the count (overrides SUMMARY_RATIO)
SUMMARY_MIN_CHARS = 20        # shorter sentences are ignored (library default 40)
SUMMARY_MAX_CHARS = 1500      # longer sentences are ignored (library default 600)

PAD = 0.4
MERGE_GAP = 2.5
#-----------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device.upper()} | Apple Silicon: {IS_APPLE_SILICON}")

[Imports] 0.0 s
Device: MPS | Apple Silicon: True


## 2. Read the video and extract audio

In [12]:
def ffmpeg_run(args, check=True):
    r = subprocess.run([FFMPEG, "-hide_banner", "-nostdin", *map(str, args)], capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr.decode(errors="ignore")[-2000:])
    return r

def probe(path):
    err = subprocess.run([FFMPEG, "-hide_banner", "-i", str(path)], capture_output=True, text=True).stderr
    m = re.search(r"Duration: (\d+):(\d+):([\d.]+)", err)
    if not m:
        raise ValueError(f"Could not read {path}:\n{err[-800:]}")
    return {"duration": int(m[1]) * 3600 + int(m[2]) * 60 + float(m[3]),
            "has_audio": "Audio:" in err}

def load_audio_16k(path):
    """Decode the audio track straight into a 16 kHz mono float32 array (no WAV file, no MoviePy)."""
    r = ffmpeg_run(["-v", "error", "-i", path, "-map", "0:a:0", "-vn",
                    "-af", "aresample=async=1:first_pts=0", "-ac", "1", "-ar", "16000",
                    "-f", "s16le", "-"])
    return np.frombuffer(r.stdout, dtype=np.int16).astype(np.float32) / 32768.0

with timed("Probe video"):
    info = probe(VIDEO_PATH)
    if not info["has_audio"]:
        raise ValueError("This video has no audio track.")
    video_duration = info["duration"]
    print(f"Video duration: {video_duration:.2f} s")

# The cache is tied to this exact file (path, size, modification time) and the Whisper settings.
_st = os.stat(VIDEO_PATH)
_key = f"{os.path.abspath(VIDEO_PATH)}|{_st.st_size}|{_st.st_mtime_ns}|{WHISPER_MODEL}|{LANGUAGE}"
cache_file = OUTPUT_DIR / f"transcript_cache_{hashlib.md5(_key.encode()).hexdigest()[:12]}.json"
use_cache = USE_TRANSCRIPT_CACHE and cache_file.exists()

if use_cache:
    print("Cached transcript found: skipping audio extraction and transcription.")
else:
    with timed("Extract audio"):
        audio_array = load_audio_16k(VIDEO_PATH)
        print(f"{len(audio_array) / 16000:.1f} s of audio loaded")

Video duration: 5425.50 s
[Probe video] 0.1 s
5425.5 s of audio loaded
[Extract audio] 2.5 s


## 3. Transcribe with word timestamps

In [13]:
def transcribe_mlx(audio):
    import mlx_whisper
    res = mlx_whisper.transcribe(audio, path_or_hf_repo=WHISPER_MODEL, word_timestamps=True,
                                 language=LANGUAGE, condition_on_previous_text=False, verbose=False)
    chunks = [{"text": w["word"], "timestamp": [float(w["start"]), float(w["end"])]}
              for seg in res["segments"] for w in seg.get("words", [])]
    return res["text"].strip(), chunks

def transcribe_hf(audio):
    """Fallback for machines without Apple Silicon (Linux, Kaggle, Windows)."""
    from transformers import pipeline
    asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=device,
                   dtype=torch.float16 if device != "cpu" else torch.float32)
    out = asr({"raw": audio, "sampling_rate": 16000}, return_timestamps="word",
              chunk_length_s=30, batch_size=8,
              generate_kwargs={"language": LANGUAGE} if LANGUAGE else {})
    return out["text"].strip(), [{"text": c["text"], "timestamp": list(c["timestamp"])} for c in out["chunks"]]

if use_cache:
    _data = json.loads(cache_file.read_text())
    full_text, transcription_chunks = _data["text"], _data["chunks"]
else:
    with timed("Transcribe"):
        full_text, transcription_chunks = (transcribe_mlx if IS_APPLE_SILICON else transcribe_hf)(audio_array)
    cache_file.write_text(json.dumps({"text": full_text, "chunks": transcription_chunks}))
    del audio_array

timestamp_level = "word"
print(f"{len(transcription_chunks)} words")
for chunk in transcription_chunks[:10]:
    print(f"[{chunk['timestamp'][0]}s - {chunk['timestamp'][1]}s]: {chunk['text']}")

 97%|█████████▋| 526018/542550 [02:55<00:05, 2993.57frames/s]

[Transcribe] 175.8 s
7941 words
[8.06s - 8.46s]:  Ron
[8.46s - 8.84s]:  Clark,
[9.22s - 9.6s]:  ready
[9.6s - 9.78s]:  for
[9.78s - 9.9s]:  your
[9.9s - 10.14s]:  first
[10.14s - 10.4s]:  day?
[11.1s - 11.48s]:  Yes,
[11.6s - 11.72s]:  ma
[11.72s - 11.9s]: 'am.


## 4. BERT extractive summary

The summarizer's own embedding loop runs one sentence at a time on the CPU with gradient tracking. `fast_create_matrix` computes **the same embeddings** (same layer, same token averaging, no special tokens) in padded batches on the GPU with gradients off. The clustering and sentence selection are unchanged.

In [14]:
_original_create_matrix = BertEmbedding.create_matrix

def fast_create_matrix(self, content, hidden=-2, reduce_option="mean", hidden_concat=False):
    if not isinstance(hidden, int) or reduce_option != "mean":
        return _original_create_matrix(self, content, hidden, reduce_option, hidden_concat)
    dev = next(self.model.parameters()).device
    pad_id = self.tokenizer.pad_token_id or 0
    token_ids = [self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(t))[:512] for t in content]
    order = sorted(range(len(content)), key=lambda i: len(token_ids[i]))  # similar lengths -> less padding
    result = [None] * len(content)
    with torch.inference_mode():
        for b in range(0, len(order), EMBED_BATCH_SIZE):
            idx = order[b:b + EMBED_BATCH_SIZE]
            width = max(1, max(len(token_ids[i]) for i in idx))
            ids = torch.full((len(idx), width), pad_id, dtype=torch.long)
            mask = torch.zeros((len(idx), width), dtype=torch.long)
            for r, i in enumerate(idx):
                n = len(token_ids[i])
                if n:
                    ids[r, :n] = torch.tensor(token_ids[i], dtype=torch.long)
                    mask[r, :n] = 1
            out = self.model(input_ids=ids.to(dev), attention_mask=mask.to(dev), output_hidden_states=True)
            h = out.hidden_states[hidden]
            m = mask.to(dev).unsqueeze(-1).to(h.dtype)
            emb = ((h * m).sum(1) / m.sum(1).clamp(min=1)).float().cpu().numpy()
            for r, i in enumerate(idx):
                result[i] = emb[r]
    return np.stack(result) if result else np.zeros((0, self.model.config.hidden_size), dtype=np.float32)

BertEmbedding.create_matrix = fast_create_matrix

with timed("Load summarizer"):
    summarizer_model = Summarizer(model=SUMMARIZER_MODEL)
    embedder = summarizer_model.model.func          # the BertEmbedding inside the summarizer
    embedder.model.to(device)
    embedder.device = torch.device(device)

with timed("Summarize"):
    summary_sentences = summarizer_model(full_text, ratio=SUMMARY_RATIO, return_as_list=True)
summary_text = " ".join(summary_sentences)

print(f"\nOriginal Length: {len(full_text)} characters")
print(f"Summary Length: {len(summary_text)} characters ({len(summary_sentences)} sentences)")
print("\nGenerated Summary:")
print(summary_text)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10111.24it/s]
[transformers] BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Load summarizer] 6.3 s
[Summarize] 1.6 s

Original Length: 40847 characters
Summary Length: 7065 characters (110 sentences)

Generated Summary:
I'll be with you in a minute, and I'll introduce you to your class. It's time I started living up to my own words. This is Radio Roanoke where the sun is shining The temperature is 78 degrees. And then the night sounds start to whisper. You're c. A. A. A. A. A. A. A. A. A. A. A. A. Good morning. You are going to change before your first class, right? You're lucky you got detention and not an assault charge. We divide each grade into four classes according to their achievement scores. These students have problems with learning, discipline, social skills, like Tejon Mitchell. I'm gonna start meeting my kids tomorrow. I'd like to meet all the parents, Ms. Vasquez, before I start teaching. I was just saying, you can help out by working with Julio every night on his homework. Together, we can get him excited about learning. You know, I've never had

## 5. Map summary sentences to video times

In [15]:
def normalize(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def map_sentences_to_segments(sentences, chunks, duration, pad=PAD, merge_gap=MERGE_GAP):
    full, spans = "", []   # spans: (char_start, char_end, chunk_index)
    for i, ch in enumerate(chunks):
        t = normalize(ch["text"])
        if not t:
            continue
        if full:
            full += " "
        spans.append((len(full), len(full) + len(t), i))
        full += t
    span_starts = [s for s, _, _ in spans]
    padded_full = f" {full} "

    def find_range(target, cursor):
        padded_target = f" {target} "
        for start_at in (cursor, 0):
            idx = padded_full.find(padded_target, start_at)
            if idx >= 0:
                return idx, idx + len(target)
        tw = target.split(" ")
        n = min(5, len(tw))
        head, tail = " ".join(tw[:n]), " ".join(tw[-n:])
        h = padded_full.find(f" {head} ", cursor)
        if h < 0:
            h = padded_full.find(f" {head} ")
        if h < 0:
            return None
        t = padded_full.find(f" {tail} ", h)
        if t < 0 or t - h > len(target) * 2:
            return None
        return h, t + len(tail)

    segments, unmatched, cursor = [], [], 0
    for sentence in sentences:
        target = normalize(sentence)
        if not target:
            continue
        found = find_range(target, cursor)
        if found is None:
            unmatched.append(sentence)
            continue
        c_start, c_end = found
        cursor = c_end
        k0 = max(0, bisect.bisect_right(span_starts, c_start) - 1)
        k1 = max(k0, bisect.bisect_left(span_starts, c_end) - 1)
        start = chunks[spans[k0][2]]["timestamp"][0]
        end = chunks[spans[k1][2]]["timestamp"][1]
        if start is None:
            unmatched.append(sentence)
            continue
        if end is None:
            end = duration
        segments.append([max(0.0, start - pad), min(duration, end + pad)])

    merged = []
    for s, e in sorted(segments):
        if merged and s - merged[-1][1] <= merge_gap:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return [{"start": s, "end": e} for s, e in merged if e - s > 0.05], unmatched

with timed("Map sentences"):
    selected_segments, unmatched_sentences = map_sentences_to_segments(
        summary_sentences, transcription_chunks, video_duration)

print(f"Mapped {len(summary_sentences) - len(unmatched_sentences)} of {len(summary_sentences)} "
      f"summary sentences into {len(selected_segments)} video segments.")
for seg in selected_segments:
    print(f"  {seg['start']:.2f}s - {seg['end']:.2f}s")
if unmatched_sentences:
    print("Could not locate these sentences in the transcript:")
    for s in unmatched_sentences:
        print("  -", s)

[Map sentences] 0.0 s
Mapped 110 of 110 summary sentences into 96 video segments.
  11.94s - 14.98s
  195.28s - 198.04s
  230.38s - 241.88s
  311.92s - 327.96s
  464.36s - 467.18s
  563.06s - 566.48s
  605.96s - 610.62s
  625.12s - 632.90s
  723.66s - 725.96s
  741.42s - 746.12s
  766.76s - 778.38s
  813.98s - 816.42s
  862.26s - 865.60s
  889.84s - 895.34s
  1074.82s - 1078.16s
  1088.78s - 1099.32s
  1263.72s - 1272.30s
  1276.08s - 1279.52s
  1307.36s - 1317.44s
  1349.70s - 1370.94s
  1392.42s - 1400.56s
  1413.30s - 1420.88s
  1565.04s - 1572.32s
  1640.56s - 1646.64s
  1687.38s - 1690.40s
  1864.98s - 1867.84s
  1877.20s - 1884.56s
  1911.98s - 1936.62s
  2009.68s - 2017.62s
  2045.18s - 2050.68s
  2129.36s - 2133.18s
  2218.50s - 2221.40s
  2238.74s - 2241.50s
  2273.94s - 2277.54s
  2293.36s - 2295.70s
  2355.70s - 2361.72s
  2373.60s - 2380.24s
  2395.74s - 2409.10s
  2420.62s - 2426.94s
  2430.50s - 2436.52s
  2460.80s - 2464.16s
  2467.14s - 2472.06s
  2597.30s - 2599.82s
  

## 6. Render the summary video with subtitles

- **Parallel cuts:** each segment is cut by its own FFmpeg process.
- **Hardware on Mac:** the Apple VideoToolbox encoder and decoder are used when available, with libx264 as the fallback.
- **Joining:** the clips are joined without re-encoding, and the subtitle track is added in the same step.
- **Subtitle timing:** uses each clip's real duration, so captions don't drift.

In [16]:
def srt_time(t):
    ms = int(round(max(0.0, t) * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def build_captions(chunks, segments, clip_durations, level, max_words=8, max_seconds=3.5):
    captions, offset = [], 0.0
    for seg, clip_len in zip(segments, clip_durations):
        s0, s1 = seg["start"], seg["end"]
        inside = []
        for ch in chunks:
            a, b = ch["timestamp"]
            if a is None:
                continue
            b = s1 if b is None else b
            if b > s0 and a < s1 and ch["text"].strip():
                inside.append((max(a, s0) - s0 + offset, min(b, s1) - s0 + offset, ch["text"].strip()))
        if level == "word":
            line = []
            def flush():
                if line:
                    captions.append((line[0][0], line[-1][1], " ".join(w for _, _, w in line)))
                    line.clear()
            for a, b, w in inside:
                if line and (len(line) >= max_words or b - line[0][0] > max_seconds):
                    flush()
                line.append((a, b, w))
                if re.search(r"[.!?]$", w):
                    flush()
            flush()
        else:
            captions.extend(inside)
        offset += clip_len
    return captions

def _works(args):
    return ffmpeg_run(args, check=False).returncode == 0

def pick_video_encoder():
    test = ["-v", "error", "-f", "lavfi", "-i", "color=s=64x64:d=0.2"]
    if IS_APPLE_SILICON:
        for enc in (["-c:v", "h264_videotoolbox", "-q:v", "65"],
                    ["-c:v", "h264_videotoolbox", "-b:v", "10M"]):
            if _works([*test, *enc, "-f", "null", "-"]):
                return enc
    return ["-c:v", "libx264", "-preset", "veryfast", "-crf", "20"]

def pick_hw_decode():
    if IS_APPLE_SILICON and _works(["-v", "error", "-hwaccel", "videotoolbox", "-t", "1",
                                     "-i", VIDEO_PATH, "-f", "null", "-"]):
        return ["-hwaccel", "videotoolbox"]
    return []

def cut_segment(i, seg, clips_dir, video_encoder, hw_decode):
    out = clips_dir / f"clip_{i:04d}.mp4"
    ffmpeg_run(["-v", "error", "-y", *hw_decode,
                "-ss", f"{seg['start']:.3f}", "-i", VIDEO_PATH, "-t", f"{seg['end'] - seg['start']:.3f}",
                "-map", "0:v:0", "-map", "0:a:0", *video_encoder, "-pix_fmt", "yuv420p",
                "-c:a", "aac", "-b:a", "160k", "-ar", "48000", "-ac", "2",
                "-avoid_negative_ts", "make_zero", out])
    return out

output_filename = str(OUTPUT_DIR / "summarized_output.mp4")
srt_path = OUTPUT_DIR / "summarized_output.srt"

if not selected_segments:
    print("Error: No segments were successfully mapped.")
else:
    with timed("Render clips"):
        video_encoder, hw_decode = pick_video_encoder(), pick_hw_decode()
        print("Encoder:", video_encoder[1], "| hardware decode:", bool(hw_decode))
        clips_dir = OUTPUT_DIR / "clips"
        shutil.rmtree(clips_dir, ignore_errors=True)
        clips_dir.mkdir(parents=True)
        with ThreadPoolExecutor(max_workers=RENDER_WORKERS) as pool:
            clip_paths = list(pool.map(lambda a: cut_segment(a[0], a[1], clips_dir, video_encoder, hw_decode),
                                       enumerate(selected_segments)))
        clip_durations = [probe(p)["duration"] for p in clip_paths]

    with timed("Join + subtitles"):
        captions = build_captions(transcription_chunks, selected_segments, clip_durations, timestamp_level)
        srt_path.write_text("".join(f"{i}\n{srt_time(a)} --> {srt_time(b)}\n{text}\n\n"
                                    for i, (a, b, text) in enumerate(captions, 1)), encoding="utf-8")
        list_file = OUTPUT_DIR / "concat.txt"
        list_file.write_text("".join("file '" + p.resolve().as_posix().replace("'", "'\\''") + "'\n"
                                     for p in clip_paths))
        subs = ["-i", srt_path, "-map", "0:v", "-map", "0:a", "-map", "1:0",
                "-c:s", "mov_text", "-metadata:s:s:0", "language=eng"] if captions else ["-map", "0:v", "-map", "0:a"]
        ffmpeg_run(["-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", list_file,
                    *subs, "-c:v", "copy", "-c:a", "copy",
                    "-movflags", "+faststart", output_filename])
        shutil.rmtree(clips_dir, ignore_errors=True)
        list_file.unlink(missing_ok=True)

    out_info = probe(output_filename)
    print(f"\nSuccess! Saved {output_filename} ({out_info['duration']:.1f}s from {video_duration:.1f}s), "
          f"audio track: {out_info['has_audio']}, {len(captions)} subtitle lines")

Encoder: h264_videotoolbox | hardware decode: True
[Render clips] 52.0 s
[Join + subtitles] 0.6 s

Success! Saved output/Movie/summarized_output.mp4 (560.1s from 5425.5s), audio track: True, 299 subtitle lines


## 7. Save transcript and summary

In [17]:
with timed("Save text files"):
    (OUTPUT_DIR / "transcript.txt").write_text(full_text.strip() + "\n", encoding="utf-8")
    with open(OUTPUT_DIR / "transcript_with_timestamps.txt", "w", encoding="utf-8") as f:
        for ch in transcription_chunks:
            a, b = ch["timestamp"]
            f.write(f"[{a if a is not None else '?'}s - {b if b is not None else 'end'}s] {ch['text'].strip()}\n")
    (OUTPUT_DIR / "summary.txt").write_text("\n".join(s.strip() for s in summary_sentences) + "\n", encoding="utf-8")

print("Saved in:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

display(Markdown("### Summary of the audio\n\n" + "\n\n".join(s.strip() for s in summary_sentences)))

[Save text files] 0.0 s
Saved in: /Users/vedantdesai/Documents/REVA_METCH_AI/MultiModel/CapstoneProject2/output/Movie
  summarized_output.mp4  (153171.9 KB)
  summarized_output.srt  (17.4 KB)
  summary.txt  (6.9 KB)
  transcript.txt  (39.9 KB)
  transcript_cache_da3672de4fc9.json  (439.4 KB)
  transcript_with_timestamps.txt  (205.7 KB)


### Summary of the audio

I'll be with you in a minute, and I'll introduce you to your class.

It's time I started living up to my own words.

This is Radio Roanoke where the sun is shining The temperature is 78 degrees.

And then the night sounds start to whisper.

You're c. A. A. A. A. A. A. A. A. A. A. A. A. Good morning.

You are going to change before your first class, right?

You're lucky you got detention and not an assault charge.

We divide each grade into four classes according to their achievement scores.

These students have problems with learning, discipline, social skills, like Tejon Mitchell.

I'm gonna start meeting my kids tomorrow.

I'd like to meet all the parents, Ms. Vasquez, before I start teaching.

I was just saying, you can help out by working with Julio every night on his homework.

Together, we can get him excited about learning.

You know, I've never had a student from the same school.

It's Saturday fooling... Shamika, who is it?

I already got a job, Mr. I got two jobs and four kids and a brother up a roll.

This means that this year is going to be different.

I am going to be your family and you are going to be my family and that means that I am not leaving and you are not leaving.

You know, school's supposed to be exciting.

Now you sound like one of those dumbass commercials that used to come on out the cartoons on Saturday morning.

Yeah, well, if I was your teacher, you would have loved it.

Not to mention my cover of my weekend shift because I'm going to LA with Jason Friend.

He's gonna meet some producers, you know, he's trying to get out of daytime heartthrob thing.

Before we start today, I would like to go over three rules for this class.

If you follow them, we will all have a lot of fun.

If you do not follow them, there will be consequences.

You all remember rule number one, we are a family.

If you are asked a question, you will say yes sir or no sir, not yep, nope, uh-uh, na-uh.

Rule number three, we will form a line to enter an exit class and go to lunch.

But because we are all a family, if one of us doesn't get in line for the lunch room, then none of us gets lunch.

All right, we're all gonna stand here until Shemeka decides to tell the truth.

And families treat each other with respect.

Tell them that's how to do this and do that.

All I'm saying is, this is our school, right?

Yeah, cause he's just gonna leave anyways.

Yo yo let's go let's go yo yo let's go let's go Hey hey Ron Park must go hey hey Ron Park must go Go yo yo yo yo yo yo yo yo yo yo yo yo yo yo yo yo Strawberry shortcake cream on top Tell me the name of your sweet pot Is it A?

You walk out now, they'll expel you for sure.

Because I would miss your glowing personality.

I mean, Jason would help, but you've got the flybook car, so... This is supposed to be well done.

I forgot to say, yo, sir, guess I get a check.

All right, let's go see some damn things.

We walked the entire island in a monsoon.

These kids can do anything I'm asking them to do more.

How long has it been since you've auditioned?

Within these four walls you can be strangers or you can be a family.

A beginning that is better than anything you ever imagined possible.

If you are quiet and you listen, every 15 seconds, I will drink a chocolate milk.

If you can do it, you may get to see me puke.

I want you to watch the clock, and every 15 seconds, bang the jar with the ruler.

When I hear the gong, I will slam down one of these.

I guess you guys don't want to see me drink all the chocolate milk.

The verb expresses action in the sentence.

Oh, you don't think I can learn how to do that?

If I learn how to double-dutch, then you guys gotta learn something from me.

Think of it as becoming somebody else's problem.

Every single one of those kids in that class just wants your respect.

We are climbing up and up and up, and we are there, and now it's time to jump.

I want you all to get the great big picture of American history.

Now let's get down to some presidential learning.

We'll start with George Washington straight from Mount Vernon.

It's a tribute to the leaders of the USA.

It's the president's rap, alright, alright, okay, okay.

It's okay to be proud of yourself for doing well on a test.

I believe this constitutes an illegal search, sir.

Sometimes it costs, and in this case, three one-hour study sessions with me.

I ain't seen him make yourself some easy money, man.

Nah, he ain't no police, he just some fool, man.

I could bring up your math and English with some extra help.

We have to talk about you turning in your homework.

Mr. Clark, I can't work on my paper right now.

I'll make dinner and you work on your paper.

I moved the third paragraph and it sounds really good.

I come home to find him in my kitchen, cooking for my family.

Miss Wallace, can we please not talk about Shamika like she's not in the room?

I send my daughter to school to be educated.

I don't expect a teacher to bring school up in my house.

She make has more potential than any kid in my class.

The white tornado here better not be teaching at this school when I bring her back tomorrow.

Please don't leave because of me, Mr. Clark.

Yeah, don't hate on me, cause I got skills.

So he thought I didn't, and I just went with it.

We are supposed to be keeping these kids in school.

Remember that I have eyes in the back of my head, so stop hiding.

Remember, there's two variables in this equation.

I can send you home but you'd be confined to bed rest for two weeks.

So, let's go over our science topics from last week.

Gravity is important because... Okay, we get it, Mr. Clark.

Very nice adjacent to let me borrow the camera.

The force of attraction between two objects.

Because... And no, Julio, it's not because the egg and the floor are hot for each other.

and your parents and yourselves what you can do.

There's no way we're going to pass those tests.

These rules, they represent everything you've accomplished this year.

Believing in each other and believing in yourselves.

Compared to all that, this test, pfft, it's nothing.

Every day in this room, we are learning things far more valuable than you can get in some book.

No, I think it really... I know that it's been hard for you to fit in.

You better kick some butt on that test then, doctor.

We can do a last minute review of the test on the way.

Okay, so, when you get to the math section... Go get him.

Jason got a job and he asked me to go to LA with him.

Oh my god, Phantom of the Apagots, we're going to Broadway!

For highest achievement in science, for most outstanding achievement in mathematics, Julio Vasquez.

For exceptional artistic achievement, Taysham Mitchell.

I'm sorry to interrupt your end of the year party, Mr. Clark, but I just received a thanks from the Board of Education informing me of the results of the state exam.

These are probably not the scores you expected.

order in which I will I will now I will now to be for highest achievement in English.

First president and commander-in-chief of the revolutionaries.

We took a step and we've been doing this since.

It's a present trap, all right, all right, OK, OK.

## 8. Timing breakdown

In [18]:
total = time.perf_counter() - RUN_START
print(f"{'Step':<22}{'Seconds':>10}{'Share':>9}")
for step, secs in STEP_TIMES.items():
    print(f"{step:<22}{secs:>10.1f}{secs / total:>9.0%}")
print(f"{'Total (incl. idle)':<22}{total:>10.1f}")

Step                     Seconds    Share
Imports                      0.0       0%
Probe video                  0.1       0%
Extract audio                2.5       1%
Transcribe                 175.8      74%
Load summarizer              6.3       3%
Summarize                    1.6       1%
Map sentences                0.0       0%
Render clips                52.0      22%
Join + subtitles             0.6       0%
Save text files              0.0       0%
Total (incl. idle)         238.9
